# WOOF — Entrenamiento v1 (EfficientNetB4)
**Modelo oficial del proyecto — objetivo: 90-93%**

## ¿Por qué EfficientNetB4?

| Factor | Detalle |
|---|---|
| Parámetros | 19M (vs 3.4M de MobileNetV2) |
| Precisión base ImageNet | 83.0% top-1 |
| Tamaño servidor | ~75 MB (H5) / ~20 MB (TFLite) |
| Embeddings | Alta calidad para coseno de similitud |
| Objetivo | 90-93% top-1 en razas de perros |

## Correcciones sobre experimentos previos (MobileNetV2)

| Problema anterior | Solución aquí |
|---|---|
| CIFAR-10 como negativos (32px borroso) | Animals10 — animales reales |
| `rescale=1/255` incorrecto | `preprocess_input` de EfficientNet |
| Arquitectura techo 75-82% | EfficientNetB4 alcanza 90%+ |
| Entrenamiento en 1 fase | 3 fases separadas |

## Datasets a agregar en Kaggle (+ Add Input)
1. `stanford-dogs-dataset` — jessicali9530
2. `perros-vs-gatos` — sergiodelcarpio
3. **`animals10`** — alessiocorrado99


## 0. Verificar entorno y datasets

In [ ]:
import tensorflow as tf
import os

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {len(tf.config.list_physical_devices("GPU")) > 0}\n')

STANFORD_PATH  = '/kaggle/input/stanford-dogs-dataset/images/Images'
ANIMALS10_PATH = '/kaggle/input/animals10/raw-img'

DOGS_CATS_BASE = '/kaggle/input/perros-vs-gatos'
DOGS_DIR = None
CATS_DIR = None
for candidate in [
    f'{DOGS_CATS_BASE}/gatos_perros/training_set',
    f'{DOGS_CATS_BASE}/training_set',
    DOGS_CATS_BASE
]:
    if os.path.exists(os.path.join(candidate, 'dogs')):
        DOGS_DIR = os.path.join(candidate, 'dogs')
        CATS_DIR = os.path.join(candidate, 'cats')
        break

print('Estado de datasets:')
print(f'  Stanford Dogs:  {os.path.exists(STANFORD_PATH)}')
print(f'  Animals10:      {os.path.exists(ANIMALS10_PATH)}')
print(f'  PvsG dogs/:     {DOGS_DIR is not None} → {DOGS_DIR}')
print(f'  PvsG cats/:     {CATS_DIR is not None}')

if not os.path.exists(ANIMALS10_PATH):
    print('\n⚠️  Animals10 no encontrado.')
    print('   → Kaggle → + Add Input → busca "animals10" (alessiocorrado99)')

## 1. Configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from PIL import Image
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization

IMG_SIZE   = 380   # tamaño nativo de EfficientNetB4
BATCH_SIZE = 32
SEED       = 42

# LR por fase
LR_HEAD = 1e-3
LR_FINE = 5e-5
LR_FULL = 5e-6

# Épocas por fase
EPOCHS_HEAD = 10
EPOCHS_FINE = 20
EPOCHS_FULL = 15

VAL_SPLIT         = 0.2
MAX_NEG_PER_CLASS = 2500
CONFIDENCE_THRESH = 0.40

OUTPUT_DIR    = '/kaggle/working/'
MODEL_H5      = f'{OUTPUT_DIR}woof_model_v3.h5'
MODEL_LITE    = f'{OUTPUT_DIR}woof_model_v3.tflite'
META_FILE     = f'{OUTPUT_DIR}woof_model_v3_metadata.json'

np.random.seed(SEED)
tf.random.set_seed(SEED)
print('Configuración lista.')
print(f'Arquitectura: EfficientNetB4 @ {IMG_SIZE}x{IMG_SIZE}')
print(f'Preprocessing: preprocess_input (NO rescale=1/255)')
print(f'Fases: {EPOCHS_HEAD} + {EPOCHS_FINE} + {EPOCHS_FULL} épocas')

## 2. Cargar imágenes de perros

In [ ]:
# Stanford Dogs — 120 razas
razas_stanford = sorted(os.listdir(STANFORD_PATH))
rows_stanford  = []
for raza in razas_stanford:
    raza_dir   = os.path.join(STANFORD_PATH, raza)
    breed_name = raza.split('-', 1)[-1].lower()
    for fname in os.listdir(raza_dir):
        rows_stanford.append({
            'filepath': os.path.join(raza_dir, fname),
            'label':    breed_name
        })

df_stanford = pd.DataFrame(rows_stanford)
print(f'Stanford Dogs:  {len(df_stanford):>6,} imgs | {df_stanford["label"].nunique()} razas')

In [ ]:
# Perros vs Gatos — carpeta dogs/
df_pvsg_dogs = pd.DataFrame(columns=['filepath', 'label'])
if DOGS_DIR and os.path.exists(DOGS_DIR):
    rows_pvsg = [{'filepath': os.path.join(DOGS_DIR, f), 'label': 'dog_generic'}
                 for f in os.listdir(DOGS_DIR)
                 if os.path.isfile(os.path.join(DOGS_DIR, f))]
    df_pvsg_dogs = pd.DataFrame(rows_pvsg)
    print(f'PvsG dogs/:     {len(df_pvsg_dogs):>6,} imgs')
else:
    print('PvsG dogs/: no encontrado')

# Animals10 — cane (perro en italiano) → dog_generic
rows_a10_dogs = []
cane_dir = os.path.join(ANIMALS10_PATH, 'cane')
if os.path.exists(cane_dir):
    for fname in os.listdir(cane_dir):
        fpath = os.path.join(cane_dir, fname)
        if os.path.isfile(fpath):
            rows_a10_dogs.append({'filepath': fpath, 'label': 'dog_generic'})
    print(f'Animals10 cane: {len(rows_a10_dogs):>6,} imgs  (→ dog_generic)')

df_dogs = pd.concat(
    [df_stanford, df_pvsg_dogs, pd.DataFrame(rows_a10_dogs)],
    ignore_index=True
)
print(f'\nTotal perros:   {len(df_dogs):>6,} imgs')

## 3. Negativos de alta calidad — Animals10 + gatos reales
> **Sin CIFAR-10** — imágenes 32px estiradas a 380px eran la causa raíz
> de que el modelo no reconociera perros con confianza suficiente.

In [ ]:
# Animals10 clases no-perro → not_a_dog
# gatto, cavallo, farfalla, pecora, ragno, gallina, elefante, scoiattolo, mucca
EXCLUDE_CLASSES = {'cane'}

rows_animals10  = []
clases_cargadas = []

if os.path.exists(ANIMALS10_PATH):
    for clase in sorted(os.listdir(ANIMALS10_PATH)):
        if clase.lower() in EXCLUDE_CLASSES:
            continue
        clase_dir = os.path.join(ANIMALS10_PATH, clase)
        if not os.path.isdir(clase_dir):
            continue
        files = [f for f in os.listdir(clase_dir)
                 if os.path.isfile(os.path.join(clase_dir, f))]
        if len(files) > MAX_NEG_PER_CLASS:
            rng   = np.random.RandomState(SEED)
            files = rng.choice(files, MAX_NEG_PER_CLASS, replace=False).tolist()
        for fname in files:
            rows_animals10.append({
                'filepath': os.path.join(clase_dir, fname),
                'label':    'not_a_dog'
            })
        clases_cargadas.append((clase, len(files)))

    print('Animals10 — clases not_a_dog:')
    for cls, n in clases_cargadas:
        print(f'  {cls:<15} {n:>5,} imgs')
    print(f'  Total:          {len(rows_animals10):>6,}')
else:
    print('⚠️  Animals10 no encontrado — agrégalo antes de continuar')

df_animals10 = pd.DataFrame(rows_animals10)

In [ ]:
# Gatos reales — Perros vs Gatos cats/
rows_cats = []
if CATS_DIR and os.path.exists(CATS_DIR):
    rows_cats = [{'filepath': os.path.join(CATS_DIR, f), 'label': 'not_a_dog'}
                 for f in os.listdir(CATS_DIR)
                 if os.path.isfile(os.path.join(CATS_DIR, f))]
    print(f'Gatos (cats/):  {len(rows_cats):>6,} imgs')
else:
    print('cats/: no encontrado')

df_cats_neg = pd.DataFrame(rows_cats)
df_neg = pd.concat([df_animals10, df_cats_neg], ignore_index=True)

print(f'\nTotal negativos: {len(df_neg):>6,}')
print(f'  Animals10:     {len(df_animals10):>6,}')
print(f'  Gatos reales:  {len(df_cats_neg):>6,}')
print('✓ Sin CIFAR-10')

## 4. Dataset combinado

In [ ]:
df_all      = pd.concat([df_dogs, df_neg], ignore_index=True)
NUM_CLASSES = df_all['label'].nunique()
ratio       = len(df_dogs) / max(len(df_neg), 1)

print('── Resumen dataset ────────────────────────────────────')
print(f'  Stanford Dogs:            {len(df_stanford):>7,}')
print(f'  PvsG dogs/:               {len(df_pvsg_dogs):>7,}')
print(f'  Animals10 cane (dogs):    {len(rows_a10_dogs):>7,}')
print(f'  Animals10 (9 clases neg): {len(df_animals10):>7,}')
print(f'  Gatos reales (neg):       {len(df_cats_neg):>7,}')
print(f'  ──────────────────────────────────────────────────')
print(f'  TOTAL:                    {len(df_all):>7,}')
print(f'  Clases:                   {NUM_CLASSES:>7}')
print(f'  Ratio perros/neg:         {ratio:.1f}:1')

df_train, df_val = train_test_split(
    df_all, test_size=VAL_SPLIT,
    stratify=df_all['label'], random_state=SEED
)
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
print(f'\nTrain: {len(df_train):,} | Val: {len(df_val):,}')

## 5. Generadores — preprocessing correcto para EfficientNetB4
> EfficientNetB4 necesita `preprocess_input`, **no** `rescale=1/255`.  
> `preprocess_input` espera píxeles en `[0, 255]` y normaliza internamente a `[-1, 1]`.

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=40,
    horizontal_flip=True,
    width_shift_range=0.25,
    height_shift_range=0.25,
    zoom_range=0.20,
    shear_range=0.15,
    brightness_range=[0.6, 1.4],
    fill_mode='nearest'
)
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_dataframe(
    dataframe=df_train, x_col='filepath', y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True, seed=SEED
)
val_gen = val_datagen.flow_from_dataframe(
    dataframe=df_val, x_col='filepath', y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

idx_to_class = {v: k for k, v in train_gen.class_indices.items()}
not_dog_idx  = train_gen.class_indices.get('not_a_dog', -1)
steps_train  = len(df_train) // BATCH_SIZE
steps_val    = len(df_val)   // BATCH_SIZE

print(f'Clases:          {len(train_gen.class_indices)}')
print(f'not_a_dog index: {not_dog_idx}')
print(f'Steps train:     {steps_train} | Steps val: {steps_val}')
print('Preprocessing:   EfficientNet preprocess_input ✓')

## 6. Modelo — EfficientNetB4 desde ImageNet

In [ ]:
# EfficientNetB4: 19M params, ImageNet top-1 83.0%
base_model = EfficientNetB4(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False  # congelar todo al inicio — Fase 1

# Cabeza clasificadora
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(NUM_CLASSES, activation='softmax', name='predictions')(x)

model = Model(inputs=base_model.input, outputs=output)

total_p     = sum(p.numpy().size for p in model.weights)
trainable_p = sum(p.numpy().size for p in model.trainable_weights)
print(f'EfficientNetB4 cargado (ImageNet)')
print(f'Parámetros totales:     {total_p:>12,}')
print(f'Parámetros entrenables: {trainable_p:>12,}  (solo cabeza — Fase 1)')
print(f'Clases de salida:       {NUM_CLASSES}')

In [ ]:
def compile_model(lr):
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=[
            'accuracy',
            tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc')
        ]
    )

## 7. FASE 1 — Solo la cabeza (base completamente congelada)
> Estabiliza los gradientes antes de tocar los pesos preentrenados de ImageNet.

In [ ]:
compile_model(LR_HEAD)

cb_head = [
    ModelCheckpoint(MODEL_H5, monitor='val_accuracy',
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=4,
                  restore_best_weights=True, verbose=1)
]

print('=== FASE 1 — Solo cabeza (base congelada) ===')
print(f'LR: {LR_HEAD} | Épocas máx: {EPOCHS_HEAD}\n')

history_head = model.fit(
    train_gen,
    steps_per_epoch=steps_train,
    epochs=EPOCHS_HEAD,
    validation_data=val_gen,
    validation_steps=steps_val,
    callbacks=cb_head,
    verbose=1
)

best_head = max(history_head.history['val_accuracy'])
print(f'\nFase 1 completada — mejor val_accuracy: {best_head:.1%}')

## 8. FASE 2 — Fine-tune del 70% superior de EfficientNetB4
> Con la cabeza estabilizada, se adaptan las capas altas para reconocimiento de razas.

In [ ]:
# Descongelar 70% superior (congelar 30% inferior)
freeze_until = int(len(base_model.layers) * 0.30)
for i, layer in enumerate(base_model.layers):
    layer.trainable = (i >= freeze_until)

trainable_now = sum(1 for l in model.layers if l.trainable)
print(f'Base layers:  {len(base_model.layers)}')
print(f'Congeladas:   primeras {freeze_until} (30%)')
print(f'Entrenables:  {trainable_now} capas del modelo total')

compile_model(LR_FINE)

cb_fine = [
    ModelCheckpoint(MODEL_H5, monitor='val_accuracy',
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=6,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.4,
                      patience=3, min_lr=1e-7, verbose=1)
]

print(f'\n=== FASE 2 — Fine-tune top 70% ===')
print(f'LR: {LR_FINE} | Épocas máx: {EPOCHS_FINE}\n')

history_fine = model.fit(
    train_gen,
    steps_per_epoch=steps_train,
    epochs=EPOCHS_FINE,
    validation_data=val_gen,
    validation_steps=steps_val,
    callbacks=cb_fine,
    verbose=1
)

best_fine = max(history_fine.history['val_accuracy'])
print(f'\nFase 2 completada — mejor val_accuracy: {best_fine:.1%}')

## 9. FASE 3 — Fine-tune completo (todo el modelo)
> LR muy bajo para no destruir los detectores de bajo nivel aprendidos con ImageNet.

In [ ]:
for layer in model.layers:
    layer.trainable = True
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False  # BN congelado evita overfitting en full fine-tune

total_trainable = sum(1 for l in model.layers if l.trainable)
bn_frozen = sum(1 for l in model.layers if isinstance(l, tf.keras.layers.BatchNormalization))
print(f'Capas entrenables: {total_trainable} / {len(model.layers)}')
print(f'BatchNorm congeladas: {bn_frozen} (previene overfitting)')

compile_model(LR_FULL)

cb_full = [
    ModelCheckpoint(MODEL_H5, monitor='val_accuracy',
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=3,  # era 5, mas agresivo contra overfitting
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                      patience=2, min_lr=1e-8, verbose=1)  # patience 3→2, reacciona antes
]

print(f'\n=== FASE 3 — Fine-tune completo (BN congelado) ===')
print(f'LR: {LR_FULL} | Épocas máx: {EPOCHS_FULL}\n')

history_full = model.fit(
    train_gen,
    steps_per_epoch=steps_train,
    epochs=EPOCHS_FULL,
    validation_data=val_gen,
    validation_steps=steps_val,
    callbacks=cb_full,
    verbose=1
)

best_full = max(history_full.history['val_accuracy'])
print(f'\nFase 3 completada — mejor val_accuracy: {best_full:.1%}')
print(f'\nResumen fases:')
print(f'  Fase 1 (cabeza):       {best_head:.1%}')
print(f'  Fase 2 (fine 70%):     {best_fine:.1%}')
print(f'  Fase 3 (full):         {best_full:.1%}')

## 10. Curvas de entrenamiento (3 fases)

In [ ]:
def concat_hist(key):
    return (history_head.history[key] +
            history_fine.history[key] +
            history_full.history[key])

all_acc      = concat_hist('accuracy')
all_val_acc  = concat_hist('val_accuracy')
all_loss     = concat_hist('loss')
all_val_loss = concat_hist('val_loss')
epochs_range = range(1, len(all_acc) + 1)

p1_end = len(history_head.history['accuracy'])
p2_end = p1_end + len(history_fine.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, train_h, val_h, title in zip(
    axes,
    [all_acc,  all_loss],
    [all_val_acc, all_val_loss],
    ['Accuracy', 'Loss']
):
    ax.plot(epochs_range, train_h, 'b-o', ms=4, label='Train')
    ax.plot(epochs_range, val_h,   'r-o', ms=4, label='Val')
    ax.axvline(p1_end + 0.5, ls='--', color='gray',  alpha=0.8, label='F1→F2')
    ax.axvline(p2_end + 0.5, ls=':',  color='black', alpha=0.8, label='F2→F3')
    ax.set_title(f'EfficientNetB4 — {title}')
    ax.set_xlabel('Época')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Entrenamiento EfficientNetB4 — 3 fases', fontsize=13)
plt.tight_layout()
plt.show()

## 11. Evaluación

In [ ]:
print('Evaluando modelo...')
results = model.evaluate(val_gen, steps=steps_val, verbose=1)
loss_v3, acc_v3, top5_v3 = results

print('\n' + '='*50)
print('  RESULTADO FINAL — EfficientNetB4')
print('='*50)
print(f'  Top-1 Accuracy: {acc_v3:.2%}')
print(f'  Top-5 Accuracy: {top5_v3:.2%}')
print(f'  Loss:           {loss_v3:.4f}')
print('='*50)
print(f'\n  Umbral de confianza: {int(CONFIDENCE_THRESH*100)}%')
print(f'  Arquitectura:        EfficientNetB4 (19M params)')
print(f'  Input size:          {IMG_SIZE}x{IMG_SIZE}')

In [ ]:
# Gráfico de barras — Top-1 vs Top-5
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(['Top-1 Accuracy', 'Top-5 Accuracy'],
              [acc_v3 * 100, top5_v3 * 100],
              color=['steelblue', 'coral'], width=0.4)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', fontsize=12, fontweight='bold')
ax.set_ylim(0, 102)
ax.set_ylabel('Accuracy (%)')
ax.set_title('EfficientNetB4 — Precisión final', fontsize=13)
ax.axhline(90, ls='--', color='green', alpha=0.6, label='Objetivo 90%')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Test de detección

In [ ]:
def predecir(img_pil, threshold=CONFIDENCE_THRESH):
    """Clasifica una imagen PIL como raza de perro o not_a_dog.
    preprocess_input espera [0, 255] — NO dividir entre 255.
    """
    img_arr = np.array(img_pil.resize((IMG_SIZE, IMG_SIZE)))
    if img_arr.ndim == 2:
        img_arr = np.stack([img_arr] * 3, axis=-1)
    img_arr  = img_arr[:, :, :3].astype(np.float32)
    img_proc = preprocess_input(img_arr)
    img_proc = np.expand_dims(img_proc, 0)

    probs = model.predict(img_proc, verbose=0)[0]
    idx   = np.argmax(probs)
    conf  = float(probs[idx])
    name  = idx_to_class[idx]

    if name == 'not_a_dog' or conf < threshold:
        return 'not_a_dog', conf
    return name, conf

In [ ]:
# Test: no-perros (deben decir not_a_dog)
print('=== TEST: imágenes no-perro ===')
test_neg = df_neg.sample(min(10, len(df_neg)), random_state=SEED)

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()
correctos = 0

for i, (_, row) in enumerate(test_neg.head(10).iterrows()):
    try:
        img          = Image.open(row['filepath']).convert('RGB')
        nombre, conf = predecir(img)
        ok = nombre == 'not_a_dog'
        if ok: correctos += 1
        axes[i].imshow(img.resize((300, 300)))
        axes[i].set_title(f'{nombre}\n({conf:.0%})',
                          color='green' if ok else 'red', fontsize=8)
    except Exception as e:
        axes[i].set_title(f'Error', fontsize=7)
    axes[i].axis('off')

plt.suptitle(f'Rechazo no-perros: {correctos}/10 — umbral {int(CONFIDENCE_THRESH*100)}%',
             fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Test: perros reales (deben reconocer la raza)
print('=== TEST: perros reales ===')
primera_raza = os.listdir(STANFORD_PATH)[0]
dog_files    = os.listdir(os.path.join(STANFORD_PATH, primera_raza))[:10]

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()
reconocidos = 0

for i, fname in enumerate(dog_files):
    try:
        img          = Image.open(os.path.join(STANFORD_PATH, primera_raza, fname)).convert('RGB')
        nombre, conf = predecir(img)
        ok = nombre != 'not_a_dog'
        if ok: reconocidos += 1
        axes[i].imshow(img.resize((300, 300)))
        axes[i].set_title(f'{nombre[:20]}\n({conf:.0%})',
                          color='green' if ok else 'red', fontsize=8)
    except Exception as e:
        axes[i].set_title(f'Error', fontsize=7)
    axes[i].axis('off')

plt.suptitle(f'Reconocimiento perros: {reconocidos}/10', fontsize=13)
plt.tight_layout()
plt.show()

## 13. Exportar modelo

In [ ]:
model.save(MODEL_H5)
size_h5 = os.path.getsize(MODEL_H5) / 1024**2
print(f'woof_model_v3.h5     → {size_h5:.1f} MB')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open(MODEL_LITE, 'wb') as f:
    f.write(tflite_model)
size_lite = os.path.getsize(MODEL_LITE) / 1024**2
print(f'woof_model_v3.tflite → {size_lite:.1f} MB')

metadata = {
    'model_name':           'woof_model_v3',
    'version':              '3.0.0',
    'architecture':         'EfficientNetB4',
    'input_size':           IMG_SIZE,
    'preprocessing':        'efficientnet.preprocess_input — input [0,255] NOT /255',
    'num_classes':          NUM_CLASSES,
    'class_names':          idx_to_class,
    'not_a_dog_class_idx':  not_dog_idx,
    'confidence_threshold': CONFIDENCE_THRESH,
    'val_accuracy':         round(float(acc_v3), 4),
    'top5_accuracy':        round(float(top5_v3), 4),
    'datasets': [
        f'Stanford Dogs ({len(df_stanford)} imgs)',
        f'PvsG dogs/ ({len(df_pvsg_dogs)} imgs)',
        f'Animals10 cane ({len(rows_a10_dogs)} imgs → dog_generic)',
        f'Animals10 non-dog ({len(df_animals10)} imgs → not_a_dog)',
        f'PvsG cats/ ({len(df_cats_neg)} imgs → not_a_dog)',
        'CIFAR-10: NO USADO'
    ],
    'training_phases': {
        'phase1_head':    f'{EPOCHS_HEAD} epocas max, LR={LR_HEAD}, base congelada',
        'phase2_fine_70': f'{EPOCHS_FINE} epocas max, LR={LR_FINE}, top 70% base',
        'phase3_full':    f'{EPOCHS_FULL} epocas max, LR={LR_FULL}, todo el modelo'
    },
    'label_smoothing': 0.1,
    'h5_size_mb':      round(size_h5, 2),
    'tflite_size_mb':  round(size_lite, 2)
}

with open(META_FILE, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'\n✓ Entrenamiento completado.')
print(f'  Top-1: {acc_v3:.2%}  |  Top-5: {top5_v3:.2%}')
print(f'  Umbral: {int(CONFIDENCE_THRESH*100)}%')
print(f'  Archivos: woof_model_v3.h5 / .tflite / _metadata.json')

## Archivos generados

| Archivo | Uso |
|---|---|
| `woof_model_v3.h5` | Backend Python — detección + embeddings |
| `woof_model_v3.tflite` | Opcional — versión compacta |
| `woof_model_v3_metadata.json` | Configuración para el backend |

### Nota de preprocessing para el backend

```python
from tensorflow.keras.applications.efficientnet import preprocess_input

# img_array en rango [0, 255] — NO dividir entre 255
img_processed = preprocess_input(img_array.astype(np.float32))
img_processed = np.expand_dims(img_processed, axis=0)

# Clasificación (¿es un perro?)
prediccion = model.predict(img_processed)

# Embedding para coseno de similitud
# (usar modelo sin cabeza clasificadora)
embedding = embedding_model.predict(img_processed)
```

### Próxima versión — v4 (si se necesita más precisión)
- Partir desde `woof_model_v3.h5` (misma arquitectura EfficientNetB4)
- Agregar más datos de razas con pocas imágenes
- Ajustar learning rates según curvas de v3
